# SolarFCD — Solar Panel Fault Classification

Reference notebook for the paper. It reproduces the full pipeline:
data loading → model construction → training → evaluation.

**Before running:** download the dataset once with

```bash
python src/download_dataset.py --out data
```

then set `DATA_ROOT` in the configuration cell below (or export the
`SOLARFCD_ROOT` environment variable). No absolute paths are hard-coded.

Class order is fixed as
`['Electrical_Fault', 'Healthy', 'Structural_Fault', 'Surface_Obstruction']`
so label indices are identical on every machine.

The command-line equivalent of this notebook is `src/train.py` + `src/evaluate.py`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import numpy as np
import time, pickle, glob, os, shutil, cv2
import random
from tensorflow.keras.optimizers import *
from tensorflow.keras.metrics import *
from tensorflow.keras.preprocessing.image import *
from tensorflow.keras import layers, models
import tensorflow as tf
import keras.backend as K
from tensorflow import keras
from keras.callbacks import *
from keras.models import Model
from tensorflow.keras.models import *
from keras.applications import *
from tensorflow.keras.layers import *
from sklearn import metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import cohen_kappa_score
#from vit_keras import vit
#from keras_flops import get_flops

In [ ]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

In [ ]:
import os
from pathlib import Path

# ---- EDIT THIS (or set the SOLARFCD_ROOT environment variable) -------------
DATA_ROOT = Path(os.environ.get('SOLARFCD_ROOT', 'data/SolarFCD'))
MODEL_DIR = Path(os.environ.get('SOLARFCD_MODELS', 'runs/notebook'))
# ---------------------------------------------------------------------------
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ['Electrical_Fault', 'Healthy', 'Structural_Fault', 'Surface_Obstruction']
Width_Imgs, Heigth_Imgs, Channal_Imags = 224, 224, 3
BATCH_SIZE = 64
shapeImage = (Width_Imgs, Heigth_Imgs, Channal_Imags)

assert DATA_ROOT.exists(), (
    f'{DATA_ROOT} not found — run: python src/download_dataset.py --out data')
print('DATA_ROOT :', DATA_ROOT.resolve())
print('MODEL_DIR :', MODEL_DIR.resolve())

## 1. Data loading (directory generators)

Images are streamed from disk with `ImageDataGenerator`. Training uses shift + flip
augmentation; validation and test use rescaling only, and the test generator is
never shuffled so that predictions stay aligned with `test_gen.classes`.

In [ ]:
TrainDS_Path = DATA_ROOT / "Train"
ValidDS_Path = DATA_ROOT / "Valid"
TestDS_Path  = DATA_ROOT / "Test"

In [ ]:
trg = ImageDataGenerator(
   
    width_shift_range=0.2, 
    height_shift_range=0.2,
    horizontal_flip=True, 
    vertical_flip=True,
    rescale=1. / 255
  
)
trgen = ImageDataGenerator(rescale=1. / 255)

validation_datagen = ImageDataGenerator(rescale=1./255)

test_datagen = ImageDataGenerator(rescale=1. / 255)

train_gen = trgen.flow_from_directory(
    TrainDS_Path,
    #classes = ['negative','positive'],
    #classes=['glioma', 'meningioma','notumor','pituitary'],
    #classes=['Non Diabetic', 'Diabetic TYPE1','Diabetic TYPE2'],
    #classes=['Benign','Malignant'],
    #classes=['Chickenpox', 'Cowpox','Healthy', 'HFMD','Measles','Monkeypox'],
    #classes = ['no','yes'],
    #classes = ['Colorectal cancer','Esophagitis','Pylorus'],
    #classes = ['CIN1','CIN2', 'CIN3', 'Normal'],
    #classes = ['Normal','Tumor'],
    #classes=['Chickenpox','Measles','Monkeypox','Normal'],
    #classes=['Glioma tumor', 'Meningioma tumor','Pituitary tumor'],
    #classes = ["CaS",'CoS','Gum','MC','OC','OLP','OT'],
    #classes=['Cancer','Normal'],
    #classes=['Dead','Live'],
    #classes=['Bengin', 'Malignant', 'Normal'],
    #classes=['Battery_Energy_Waste', 'Construction_Ceramic_Cement', 'Electronic','Glass', 'Medical_Hazardous', 'Metal',
     #       'Misc_Trash', 'Organic_Biodegradable', 'Others_Unique', 'Paper_Cardboard', 'Plastics',
     #       'Rubber_Polymer', 'Textiles_Fabric', 'Wood'],
    #classes=['brick', 'cob', 'concrete','decks','pavements','road','stone','title','walls'],
    #classes=['Electrical_Fault', 'Healthy', 'Structural_Fault','Surface_Obstruction'],
    #classes=['Dementia', 'Non Demented'],
    #classes=['MildDemented', 'ModerateDemented','NonDemented','VeryMildDemented'],
    classes=CLASSES,
    target_size=(Width_Imgs, Heigth_Imgs),
    batch_size=BATCH_SIZE,
    shuffle=True )

valid_gen = validation_datagen.flow_from_directory(
    ValidDS_Path,
    classes=CLASSES,
    target_size=(Width_Imgs, Heigth_Imgs),
    batch_size=BATCH_SIZE,
    shuffle=True)


test_gen = test_datagen.flow_from_directory(
    TestDS_Path,
    classes=CLASSES,
    target_size=(Width_Imgs, Heigth_Imgs),
    batch_size=BATCH_SIZE,
    shuffle=False )

In [ ]:
# ModelCheckpoint callback - save best weights
tl_checkpoint_1 = ModelCheckpoint(filepath= str(MODEL_DIR / "ViTB16.keras"),
                                  save_best_only=True,
                                  monitor = "val_accuracy",
                                  verbose=1)
reducelr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.1,
    patience=3,
    verbose=0,
    mode="auto",
    min_delta=0.00001,
    cooldown=0,
    min_lr=0)

In [ ]:
def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return "{}:{:>02}:{:>05.2f}".format(h, m, s)

In [ ]:
METRICS = [keras.metrics.CategoricalAccuracy(name='accuracy'),
           keras.metrics.Precision(name='precision'),
           keras.metrics.Recall(name='recall'),
           keras.metrics.AUC(name='auc')]

In [ ]:
def create_model(input_shape, n_classes, optimizer):

    conv_base =  VGG19(include_top=False, weights='imagenet', input_shape=input_shape)

    # Freeze the base_model #MobileNetV3Large, MobileNetV3Small
    conv_base.trainable = False

    # append my own layers on the top of the model for Transfer Learning
    x = conv_base.output
   
    x = keras.layers.Flatten()(x)

     # 1st FC layer
    x = keras.layers.Dense(n_classes, 'softmax')(x)

    model = Model(inputs=conv_base.input, outputs=x, name="D121")
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=METRICS)

    return model

In [ ]:
def residual_block(input_tensor, filters, kernel_size=(3, 3), strides=(1, 1), activation='relu'):
    x = Conv2D(filters=filters, kernel_size=kernel_size, strides=strides, padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = Activation(activation)(x)
    x = Conv2D(filters=filters, kernel_size=kernel_size, padding='same')(x)
    x = BatchNormalization()(x)
    if strides != (1, 1) or input_tensor.shape[-1] != filters:
        shortcut = Conv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')(input_tensor)
        shortcut = BatchNormalization()(shortcut)
        x = Add()([x, shortcut])
    else:
        x = Add()([x, input_tensor])
    x = Activation(activation)(x)
    return x

In [ ]:
def naive_inception_module(input_tensor, num_filters):
    # 1x1 Convolution for dimension reduction
    conv1x1 = Conv2D(num_filters, (1, 1), padding='same', activation='gelu')(input_tensor)
    conv1x1 = LeakyReLU(alpha=0.2)(conv1x1)

    # 3x3 Convolution followed by 1x1 Convolution for dimension reduction
    conv3x3 = Conv2D(num_filters, (1, 1), padding='same', activation='gelu')(input_tensor)
    conv3x3 = LeakyReLU(alpha=0.2)(conv3x3)
    conv3x3 = Conv2D(num_filters, (3, 3), padding='same', activation='gelu')(conv3x3)
    conv3x3 = LeakyReLU(alpha=0.2)(conv3x3)

    # 5x5 Convolution followed by 1x1 Convolution for dimension reduction
    conv5x5 = Conv2D(num_filters, (1, 1), padding='same', activation='gelu')(input_tensor)
    conv5x5 = LeakyReLU(alpha=0.2)(conv5x5)
    conv5x5 = Conv2D(num_filters, (5, 5), padding='same', activation='gelu')(conv5x5)
    conv5x5 = LeakyReLU(alpha=0.2)(conv5x5)

    # Max pooling followed by 1x1 Convolution for dimension reduction
    max_pool = MaxPooling2D((3, 3), strides=(1, 1), padding='same')(input_tensor)
    max_pool_conv = Conv2D(num_filters, (1, 1), padding='same', activation='gelu')(max_pool)
    max_pool_conv = LeakyReLU(alpha=0.2)(max_pool_conv)
    
  

    # Concatenate the outputs along the channel axis
    inception_module = concatenate([conv1x1, conv3x3, conv5x5, max_pool_conv], axis=-1)

    return inception_module

In [ ]:
def inception_module(x, filters):
    # 1x1 Convolution
    conv1x1 = Conv2D(filters, (1, 1), padding='same', activation='relu')(x)

    # 3x3 Convolution
    conv3x3 = Conv2D(filters, (3, 3), padding='same', activation='relu')(x)

    # 5x5 Convolution
    conv5x5 = Conv2D(filters, (5, 5), padding='same', activation='relu')(x)

    # Max Pooling
    max_pool = MaxPooling2D((3, 3), strides=(1, 1), padding='same')(x)
    max_pool_conv = Conv2D(filters, (1, 1), padding='same', activation='relu')(max_pool)

    # Concatenate the outputs
    inception_output = Concatenate()([conv1x1, conv3x3, conv5x5, max_pool_conv])

    return inception_output

In [ ]:
def channel_attention(input_feature, ratio=8):
    channel_axis = 1 if K.image_data_format() == "channels_first" else -1
    channel = input_feature.shape[channel_axis]
   
    shared_layer_one = Dense(channel//ratio,
                             activation='relu',
                             kernel_initializer='he_normal',
                             use_bias=True,
                             bias_initializer='zeros')
    shared_layer_two = Dense(channel,
                             kernel_initializer='he_normal',
                             use_bias=True,
                             bias_initializer='zeros')
   
    avg_pool = GlobalAveragePooling2D()(input_feature)    
    avg_pool = Reshape((1,1,channel))(avg_pool)
    assert avg_pool.shape[1:] == (1,1,channel)
    avg_pool = shared_layer_one(avg_pool)
    assert avg_pool.shape[1:] == (1,1,channel//ratio)
    avg_pool = shared_layer_two(avg_pool)
    assert avg_pool.shape[1:] == (1,1,channel)
   
    max_pool = GlobalMaxPooling2D()(input_feature)
    max_pool = Reshape((1,1,channel))(max_pool)
    assert max_pool.shape[1:] == (1,1,channel)
    max_pool = shared_layer_one(max_pool)
    assert max_pool.shape[1:] == (1,1,channel//ratio)
    max_pool = shared_layer_two(max_pool)
    assert max_pool.shape[1:] == (1,1,channel)
   
    cbam_feature = Add()([avg_pool,max_pool])
    cbam_feature = Activation('sigmoid')(cbam_feature)

    if K.image_data_format() == "channels_first":
        cbam_feature = Permute((3, 1, 2))(cbam_feature)
   
    return multiply([input_feature, cbam_feature])

def create_model(input_shape, n_classes, optimizer):

    conv_base = MobileNet(include_top=False, weights='imagenet', input_shape=input_shape)

    # Freeze the base_model
    conv_base.trainable = False

   

    # Append channel attention block
    attention_output = channel_attention(conv_base.output)

    # Flatten the output
    x = Flatten()(attention_output)

    # Add a dense layer for classification
    x = Dense(n_classes, activation='softmax')(x)

    # Create the model
    model = Model(inputs=conv_base.input, outputs=x, name="MBCA")

    # Compile the model
    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=METRICS)

    return model

In [ ]:
  # Define SE block function
def se_block(input_tensor, ratio=16):
    num_channels = input_tensor.shape[-1]
    squeeze = GlobalAveragePooling2D()(input_tensor)
    excitation = Dense(num_channels // ratio, activation='relu')(squeeze)
    excitation = Dense(num_channels, activation='sigmoid')(excitation)
    excitation = Reshape((1, 1, num_channels))(excitation)
    scaled_input = multiply([input_tensor, excitation])
    return scaled_input

In [ ]:
def cbam_block(cbam_feature, ratio=8):

   
    cbam_feature = channel_attention(cbam_feature, ratio)
    cbam_feature = spatial_attention(cbam_feature)
    return cbam_feature

def channel_attention(input_feature, ratio=8):
    channel_axis = 1 if K.image_data_format() == "channels_first" else -1
    channel = input_feature.shape[channel_axis]
   
    shared_layer_one = Dense(channel//ratio,
                             activation='relu',
                             kernel_initializer='he_normal',
                             use_bias=True,
                             bias_initializer='zeros')
    shared_layer_two = Dense(channel,
                             kernel_initializer='he_normal',
                             use_bias=True,
                             bias_initializer='zeros')
   
    avg_pool = GlobalAveragePooling2D()(input_feature)    
    avg_pool = Reshape((1,1,channel))(avg_pool)
    assert avg_pool.shape[1:] == (1,1,channel)
    avg_pool = shared_layer_one(avg_pool)
    assert avg_pool.shape[1:] == (1,1,channel//ratio)
    avg_pool = shared_layer_two(avg_pool)
    assert avg_pool.shape[1:] == (1,1,channel)
   
    max_pool = GlobalMaxPooling2D()(input_feature)
    max_pool = Reshape((1,1,channel))(max_pool)
    assert max_pool.shape[1:] == (1,1,channel)
    max_pool = shared_layer_one(max_pool)
    assert max_pool.shape[1:] == (1,1,channel//ratio)
    max_pool = shared_layer_two(max_pool)
    assert max_pool.shape[1:] == (1,1,channel)
   
    cbam_feature = Add()([avg_pool,max_pool])
    cbam_feature = Activation('sigmoid')(cbam_feature)

    if K.image_data_format() == "channels_first":
        cbam_feature = Permute((3, 1, 2))(cbam_feature)
   
    return multiply([input_feature, cbam_feature])

def spatial_attention(input_feature):
    kernel_size = 7
   
    if K.image_data_format() == "channels_first":
        channel = input_feature.shape[1]
        cbam_feature = Permute((2,3,1))(input_feature)
    else:
        channel = input_feature.shape[-1]
        cbam_feature = input_feature
   
    avg_pool = Lambda(lambda x: K.mean(x, axis=3, keepdims=True))(cbam_feature)
    assert avg_pool.shape[-1] == 1
    max_pool = Lambda(lambda x: K.max(x, axis=3, keepdims=True))(cbam_feature)
    assert max_pool.shape[-1] == 1
    concat = Concatenate(axis=3)([avg_pool, max_pool])
    assert concat.shape[-1] == 2
    cbam_feature = Conv2D(filters = 1,
                    kernel_size=kernel_size,
                    strides=1,
                    padding='same',
                    activation='sigmoid',
                    kernel_initializer='he_normal',
                    use_bias=False)(concat)
    assert cbam_feature.shape[-1] == 1
   
    if K.image_data_format() == "channels_first":
        cbam_feature = Permute((3, 1, 2))(cbam_feature)
       
    return multiply([input_feature, cbam_feature])

In [ ]:
def attention_block(input_tensor, num_filters):
    # Compute attention weights
    x = Conv2D(num_filters, (4, 4), padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)
    
    y = Conv2D(num_filters, (3, 3), padding='same')(x)
    y = BatchNormalization()(y)
    y = LeakyReLU(alpha=0.2)(y)
    
    z = Conv2D(num_filters, (3, 3), padding='same')(x)
    z = BatchNormalization()(z)
    z = LeakyReLU(alpha=0.2)(z)
    z = Conv2D(num_filters, (2, 2), padding='same')(z)
    z = BatchNormalization()(z)
    z = LeakyReLU(alpha=0.2)(z)
    z = Dropout(0.2)(z)


    zz = Conv2D(num_filters, (3, 3), padding='same')(input_tensor)
    zz = BatchNormalization()(zz)
    zz = LeakyReLU(alpha=0.2)(zz)
    zz = Conv2D(num_filters, (4, 4), padding='same')(zz)
    zz = BatchNormalization()(zz)
    zz = LeakyReLU(alpha=0.2)(zz)
    zz = Dropout(0.2)(zz)
    
    output1 = layers.concatenate([ y, z, zz])
    
    output2 = layers.concatenate([ output1, zz])
    
    return output2

In [ ]:
def dense_block(x, num_layers, growth_rate):

    for _ in range(num_layers):
        # Each layer is connected to all previous layers via concatenation
        y = Dense(growth_rate, activation='relu')(x)
        x = concatenate([x, y])
    
    return x

In [ ]:
def naive_inception_module(input_tensor, num_filters):
    # 1x1 Convolution for dimension reduction
    conv1x1 = Conv2D(num_filters, (1, 1), padding='same', activation='relu')(input_tensor)

    # 3x3 Convolution followed by 1x1 Convolution for dimension reduction
    conv3x3 = Conv2D(num_filters, (1, 1), padding='same', activation='relu')(input_tensor)
    conv3x3 = Conv2D(num_filters, (3, 3), padding='same', activation='relu')(conv3x3)

    # 5x5 Convolution followed by 1x1 Convolution for dimension reduction
    conv5x5 = Conv2D(num_filters, (1, 1), padding='same', activation='relu')(input_tensor)
    conv5x5 = Conv2D(num_filters, (5, 5), padding='same', activation='relu')(conv5x5)

    # Max pooling followed by 1x1 Convolution for dimension reduction
    max_pool = MaxPooling2D((3, 3), strides=(1, 1), padding='same')(input_tensor)
    max_pool_conv = Conv2D(num_filters, (1, 1), padding='same', activation='relu')(max_pool)
    
    # Concatenate the outputs along the channel axis
    inception_module = concatenate([conv1x1, conv3x3, conv5x5, max_pool_conv], axis=-1)

    return inception_module

In [ ]:
def naive_inception_moduleM(input_tensor, num_filters):
    # 1x1 Convolution for dimension reduction
    conv1x1 = Conv2D(num_filters, (1, 1), padding='same', activation='gelu')(input_tensor)
    conv1x1 = LeakyReLU(alpha=0.2)(conv1x1)

    # 3x3 Convolution followed by 1x1 Convolution for dimension reduction
    conv3x3 = Conv2D(num_filters, (1, 1), padding='same', activation='gelu')(input_tensor)
    conv3x3 = LeakyReLU(alpha=0.2)(conv3x3)
    conv3x3 = Conv2D(num_filters, (3, 3), padding='same', activation='gelu')(conv3x3)
    conv3x3 = LeakyReLU(alpha=0.2)(conv3x3)

    # 5x5 Convolution followed by 1x1 Convolution for dimension reduction
    conv5x5 = Conv2D(num_filters, (1, 1), padding='same', activation='gelu')(input_tensor)
    conv5x5 = LeakyReLU(alpha=0.2)(conv5x5)
    conv5x5 = Conv2D(num_filters, (5, 5), padding='same', activation='gelu')(conv5x5)
    conv5x5 = LeakyReLU(alpha=0.2)(conv5x5)

    # Max pooling followed by 1x1 Convolution for dimension reduction
    max_pool = MaxPooling2D((3, 3), strides=(1, 1), padding='same')(input_tensor)
    max_pool_conv = Conv2D(num_filters, (1, 1), padding='same', activation='gelu')(max_pool)
    max_pool_conv = LeakyReLU(alpha=0.2)(max_pool_conv)
    
  

    # Concatenate the outputs along the channel axis
    inception_module = concatenate([conv1x1, conv3x3, conv5x5, max_pool_conv], axis=-1)

    return inception_module

In [ ]:
def residual(input_tensor, filters, kernel_size=(3, 3), strides=(1, 1), activation='relu'):
    x = SeparableConv2D(filters=filters, kernel_size=kernel_size, strides=strides, padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = LeakyReLU(alpha=0.2)(x)  # Adjust alpha as needed
    x = SeparableConv2D(filters=filters, kernel_size=kernel_size, padding='same')(x)
    x = BatchNormalization()(x)

    y = SeparableConv2D(filters=filters, kernel_size=kernel_size, strides=strides, padding='same')(input_tensor)
    y = BatchNormalization()(y)
    
    if strides != (1, 1) or input_tensor.shape[-1] != filters:
        shortcut = SeparableConv2D(filters=filters, kernel_size=(1, 1), strides=strides, padding='same')(input_tensor)
        shortcut = BatchNormalization()(shortcut)
        x = Add()([x, shortcut])
    else:
        x = Add()([x,y , input_tensor])
    
    x = Activation(activation)(x)  # Adjust alpha as needed
    return x

In [ ]:
def create_model(input_shape, n_classes, optimizer):

    conv_base = vit.vit_b16(pretrained=True, include_top=False, pretrained_top=False, classes=n_classes, activation='softmax', image_size=(224, 224))

    x = conv_base.output
    conv_base.trainable = False

    x = conv_base.output
    x  = keras.layers.Flatten()(x)
    
     # 1st FC layer
    x = keras.layers.Dense(n_classes, 'softmax')(x)
    
    #softmax
    model = Model(inputs=conv_base.input, outputs=x, name="VIT_b16")


    model.compile(optimizer=optimizer,
                  loss='categorical_crossentropy',
                  metrics=METRICS)

    return model

In [ ]:
optim_1 = Adam(learning_rate=0.001)
n_classes= 4
n_epochs = 20
Train_model = create_model(shapeImage, n_classes, optim_1)
#print (Train_model.summary(), )

In [ ]:
start = time.time()

Train_model = Train_model.fit(train_gen , batch_size=BATCH_SIZE, epochs=n_epochs, validation_data=valid_gen , callbacks=[tl_checkpoint_1, reducelr], verbose=1)
Elapsed = time.time()-start
print (f'Training time: {hms_string(Elapsed)}')

In [ ]:
model = keras.models.load_model(str(MODEL_DIR / "D121.keras"))
#print (model.summary())

In [ ]:
start = time.time()
prediction = np.argmax(model.predict(test_gen), axis=1)
print('Test Data accuracy: ',accuracy_score(test_gen.classes, prediction)*100)
Elapsed = time.time()-start
print (f'Training time: {hms_string(Elapsed)}')

In [ ]:
print("F1-Score", f1_score(test_gen.classes, prediction, average='macro'))
print("Recall", recall_score(test_gen.classes, prediction, average='macro'))
print("Precision", precision_score(test_gen.classes, prediction, average='macro'))
from sklearn.metrics import confusion_matrix
print(confusion_matrix(test_gen.classes, prediction))
print("Cohen", cohen_kappa_score(test_gen.classes, prediction))

In [ ]:
report = classification_report(test_gen.classes, prediction, output_dict=True)

# Format the results to four decimal places
for label, metrics in report.items():
    if label not in ['accuracy', 'macro avg', 'weighted avg']:  # Ignore summary lines
        for metric, value in metrics.items():
            report[label][metric] = f"{value:.4f}"

# Print the formatted report
print("Classification Report (with 4 decimal places):")
for label, metrics in report.items():
    if label not in ['accuracy', 'macro avg', 'weighted avg']:  # Ignore summary lines
        print(f"{label}: Precision = {metrics['precision']}, Recall = {metrics['recall']}, F1-score = {metrics['f1-score']}")

In [ ]:
flops = get_flops(model, batch_size=64)
# Print FLOPS in billions (G)
print(f"FLOPS: {flops / 10 ** 9:.03} G")
# Print FLOPS in millions (M)
print(f"FLOPS: {flops / 10 ** 6:.03} M")